In [28]:
import requests
import pandas as pd

Get all event_tickers for the Serie

In [29]:
EVENTS_API = "https://api.elections.kalshi.com/trade-api/v2/events" 
#SERIES_TICKER = "KXCPIYOY" #possible tickers for inflation: KXCPI, KXCPIYOY, KXLCPIMAXYOY, KXECONSTATCPIYOY, KXCPICOREYOY, KXECONSTATCPICORE (source: https://kalshi.com/category/economics/inflation)

def get_eventtickers_from_series(series_ticker):
    series_params= {
            "series_ticker": series_ticker,
            "status": "settled",
            "limit": 200 #max
        }
    response = requests.get(EVENTS_API, params=series_params)
    events_data = response.json()['events']
    events_df = pd.DataFrame(events_data)
    return events_df['event_ticker']

Get all the markets for an event

In [30]:

HISTORICAL_API = "https://api.elections.kalshi.com/trade-api/v2/historical/markets"
#EVENT_TICKER = "CPIYOY-23MAY" # KXCPIYOY-26FEB	OF KXCPI-26FEB

def get_markets_from_event(event_ticker):
    event_params= {
        "event_ticker": event_ticker,
        "status": "settled",
        "limit": 1000 #max
    }
    response = requests.get(HISTORICAL_API, params=event_params)
    market_data = response.json()['markets']
    market_data_df = pd.DataFrame(market_data)
    return market_data_df

markets = get_markets_from_event("CPIYOY-23MAY")
display(markets.columns)



Index(['can_close_early', 'close_time', 'created_time', 'event_ticker',
       'expected_expiration_time', 'expiration_time', 'expiration_value',
       'floor_strike', 'fractional_trading_enabled', 'last_price_dollars',
       'latest_expiration_time', 'liquidity_dollars', 'market_type',
       'no_ask_dollars', 'no_bid_dollars', 'no_sub_title',
       'notional_value_dollars', 'open_interest_fp', 'open_time',
       'previous_price_dollars', 'previous_yes_ask_dollars',
       'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges',
       'response_price_units', 'result', 'rules_primary', 'rules_secondary',
       'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars',
       'status', 'strike_type', 'subtitle', 'tick_size', 'ticker', 'title',
       'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars',
       'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp',
       'yes_sub_title'],
      dtype='object')

In [31]:
import pandas as pd
import numpy as np

def generate_event_scores(df):
    """
    Analyseert een Kalshi-event dataframe. 
    Inclusief checks om KeyErrors te voorkomen als kolommen ontbreken.
    """
    # 1. Veiligheidscheck: is de dataframe leeg?
    if df is None or df.empty:
        return {
            "average_prediction": 0, "market_confidence_score": 0,
            "liquidity_score": 0, "trading_intensity": 0, "bid_ask_tightness_score": 0
        }

    df = df.copy()

    # 2. Beschikbare kolommen controleren en aanmaken indien afwezig
    # Dit voorkomt de KeyError: 'volume_fp'
    required_cols = [
        'yes_bid_dollars', 'yes_ask_dollars', 'volume_fp', 
        'floor_strike', 'last_price_dollars', 
        'yes_bid_size_fp', 'yes_ask_size_fp'
    ]
    
    for col in required_cols:
        if col not in df.columns:
            df[col] = 0  # Maak de kolom aan met nullen als hij ontbreekt
        
        # Zorg dat alles numeriek is
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # --- BEREKENINGEN ---

    # 1. Average Prediction
    total_volume = df['volume_fp'].sum()
    if total_volume > 0:
        avg_prediction = (df['floor_strike'] * df['volume_fp']).sum() / total_volume
    else:
        # Als volume 0 is, neem het ongewogen gemiddelde van de strikes
        avg_prediction = df['floor_strike'].mean() if not df['floor_strike'].empty else 0

    # 2. Market Confidence
    std_dev = df['floor_strike'].std()
    confidence_score = 100 / (1 + std_dev) if (pd.notnull(std_dev) and std_dev > 0) else 100

    # 3. Liquidity Score
    liquidity_score = df['yes_bid_size_fp'].sum() + df['yes_ask_size_fp'].sum()
    
    # 4. Trading Intensity
    trading_intensity = df['volume_fp'].mean()

    # 5. Bid-Ask Tightness
    df['spread'] = (df['yes_ask_dollars'] - df['yes_bid_dollars']).abs()
    avg_spread = df['spread'].mean()
    tightness_score = max(0, 100 * (1 - avg_spread))

    # Resultaten netjes afronden
    return {
        "average_prediction": round(float(avg_prediction), 4),
        "market_confidence_score": round(float(confidence_score), 2),
        "liquidity_score": round(float(liquidity_score), 2),
        "trading_intensity": round(float(trading_intensity), 2),
        "bid_ask_tightness_score": round(float(tightness_score), 2)
    }

Run all code

In [ ]:
event_tickers = get_eventtickers_from_series("KXCPIYOY") #must be all capitals
market_dfs = []
market_data_by_event = {}

for event_ticker in event_tickers:
    market_data = get_markets_from_event(event_ticker)
    scores = generate_event_scores(market_data)
    
    # Maak een platte dictionary voor de rij
    row = {
        "event_ticker": event_ticker
    }
    
    # Voeg alle scores toe als losse sleutels/kolommen
    row.update(scores) 
    
    market_dfs.append(row)
    market_data_by_event[event_ticker] = market_data

# Zet de lijst met dictionaries om naar een DataFrame
df_final = pd.DataFrame(market_dfs)




PermissionError: [Errno 13] Permission denied: 'kalshi_scores.xlsx'

In [33]:
# Sla op naar Excel
df_final.to_excel("kalshi_scores_v2.xlsx", index=False)
print(f"Succes! Excel opgeslagen met {len(df_final.columns)} kolommen.")

Succes! Excel opgeslagen met 6 kolommen.


In [34]:
print(df_final.head())

     event_ticker  average_prediction  market_confidence_score  \
0  KXCPIYOY-26FEB              0.0000                     0.00   
1  KXCPIYOY-26JAN              0.0000                     0.00   
2  KXCPIYOY-25DEC              0.0000                     0.00   
3  KXCPIYOY-25NOV              2.9250                    80.32   
4  KXCPIYOY-25OCT              3.0049                    80.32   

   liquidity_score  trading_intensity  bid_ask_tightness_score  
0              0.0               0.00                      0.0  
1              0.0               0.00                      0.0  
2              0.0               0.00                      0.0  
3              0.0           94594.62                      0.0  
4              0.0          112445.88                      0.0  


In [ ]:
# Voorbeeld van gebruik (uitgaande van jouw dataframe 'df_event'):
df_event = market_data_by_event["CPIYOY-23MAY"] #vervang door jouw event ticker
resultaten = generate_event_scores(df_event)
print(resultaten)

{'average_prediction': 4.0448, 'market_confidence_score': 69.1, 'liquidity_score': 0.0, 'trading_intensity': 1745.73, 'bid_ask_tightness_score': 96.53}
